# Model selection criteria: lab notebook

**Statistical Modelling and Forecasting, ARIMA and vector autoregression**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.arima_process import ArmaProcess
from statsmodels.tsa.stattools import arma_order_select_ic

### 1. One sample from an ARMA(1,1) and a grid of candidate models

In [ ]:
# 1. One sample from an ARMA(1,1) and a grid of candidate models
true = ArmaProcess([1, -0.6], [1, 0.5])
y = true.generate_sample(nsample=150, distrvs=np.random.default_rng(11).standard_normal)
rows = []
for p in range(3):
    for q in range(3):
        r = ARIMA(y, order=(p, 0, q)).fit()
        rows.append({"p": p, "q": q, "k": p + q + 2, "AIC": r.aic, "AICc": r.aicc, "BIC": r.bic})
T = pd.DataFrame(rows)
for c in ["AIC", "AICc", "BIC"]:
    T["d" + c] = T[c] - T[c].min()
print(T.round(1).sort_values("AIC").head(5).to_string(index=False))
print("chosen by AIC :", tuple(T.loc[T.AIC.idxmin(), ["p", "q"]].astype(int)))
print("chosen by BIC :", tuple(T.loc[T.BIC.idxmin(), ["p", "q"]].astype(int)))
sel = arma_order_select_ic(y, max_ar=2, max_ma=2, ic=["aic", "bic"])
print("same answer with statsmodels (AIC, BIC):", tuple(int(v) for v in sel.aic_min_order), tuple(int(v) for v in sel.bic_min_order))

### 2. How often does each criterion find the true AR(2) order? (fast least-squares fits)

In [ ]:
# 2. How often does each criterion find the true AR(2) order? (fast least-squares fits)
def ar_ic(x, pmax=6):
    n = len(x); m = pmax; out = {"aic": [], "bic": [], "aicc": []}
    for p in range(pmax + 1):
        X = np.column_stack([np.ones(n - m)] + [x[m - i - 1:n - i - 1] for i in range(p)])
        yy = x[m:]
        b = np.linalg.lstsq(X, yy, rcond=None)[0]
        sse = np.sum((yy - X @ b) ** 2); N = len(yy); k = p + 2
        l = N * np.log(sse / N)
        out["aic"].append(l + 2 * k); out["bic"].append(l + k * np.log(N)); out["aicc"].append(l + 2 * k + 2 * k * (k + 1) / (N - k - 1))
    return {c: int(np.argmin(v)) for c, v in out.items()}
ar2 = ArmaProcess([1, -0.9, 0.4], [1])
rng = np.random.default_rng(2)
for n in (60, 200, 800):
    hit = {"aic": 0, "bic": 0, "aicc": 0}; over = {"aic": 0, "bic": 0, "aicc": 0}
    for _ in range(300):
        c = ar_ic(ar2.generate_sample(nsample=n, distrvs=rng.standard_normal))
        for k in hit:
            hit[k] += c[k] == 2; over[k] += c[k] > 2
    print(f"n={n:4d}  correct p=2:", {k: round(v / 300, 2) for k, v in hit.items()}, "| too large:", {k: round(v / 300, 2) for k, v in over.items()})

### 3. The numerical problem

In [ ]:
# 3. The numerical problem
n, S, k = 50, [120, 96, 91], [2, 3, 5]
aic = [n * np.log(s / n) + 2 * kk for s, kk in zip(S, k)]
bic = [n * np.log(s / n) + kk * np.log(n) for s, kk in zip(S, k)]
print("AIC:", np.round(aic, 2), "-> model", int(np.argmin(aic)) + 1)
print("BIC:", np.round(bic, 2), "-> model", int(np.argmin(bic)) + 1)
print("AICc of model 3:", round(aic[2] + 2 * k[2] * (k[2] + 1) / (n - k[2] - 1), 2))

## Expected output

Running all cells above should print:

```
 p  q  k   AIC  AICc   BIC  dAIC  dAICc  dBIC
 1  1  4 413.3 413.6 425.4   0.0    0.0   0.0
 2  2  6 415.2 415.8 433.3   1.9    2.2   7.9
 2  1  5 415.3 415.7 430.4   2.0    2.1   5.0
 1  2  5 415.3 415.7 430.3   2.0    2.1   5.0
 0  2  4 417.4 417.7 429.4   4.1    4.1   4.1
chosen by AIC : (1, 1)
chosen by BIC : (1, 1)
same answer with statsmodels (AIC, BIC): (1, 1) (1, 1)
n=  60  correct p=2: {'aic': 0.68, 'bic': 0.81, 'aicc': 0.75} | too large: {'aic': 0.28, 'bic': 0.07, 'aicc': 0.2}
n= 200  correct p=2: {'aic': 0.72, 'bic': 0.98, 'aicc': 0.76} | too large: {'aic': 0.28, 'bic': 0.02, 'aicc': 0.24}
n= 800  correct p=2: {'aic': 0.75, 'bic': 0.99, 'aicc': 0.75} | too large: {'aic': 0.25, 'bic': 0.01, 'aicc': 0.25}
AIC: [47.77 38.62 39.94] -> model 2
BIC: [51.6  44.35 49.5 ] -> model 2
AICc of model 3: 41.31
```

## Practice exercises

Try these in the cells below.

1. Use the ARMA(1,1) truth with n = 100. Which orders do AIC and BIC choose? Press New sample ten times and count.
2. Increase n to 400. Does BIC get the true order more often?
3. Switch the truth to AR(2) and look at the (p, q) grid. Which wrong models have an AIC within 2 of the best?

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
